# Restaurant search — GPU playground (Approaches 6 & 7)

Evaluate and experiment with the GPU-heavy search approaches against the
209-query benchmark (NDCG@5 / Recall@10 / MRR):

- **Approach 6** — cross-encoder scoring every menu item (`crossencoder`)
- **Approach 7** — local LLM as reader via vLLM (`llm_search.py`)
- plus the cheap baselines (`text`, `enhanced`) and optionally ColBERT / SBERT for comparison

**Before running:** *Runtime → Change runtime type → GPU* (L4 or A100 recommended;
a T4 works with a ≤3B model, see the Approach 7 cell).
The repo is private: add a Colab secret named **`GHTOKEN`** (key icon in the left sidebar,
enable *Notebook access*) holding a GitHub fine-grained token with *Contents: read* on
`ahsank/restaurant-search`. Without a token, the notebook uses an existing checkout at
`/content/restaurant-search` if there is one (e.g. an uploaded and unzipped copy).

Run the install cell **first**, before anything imports `torch`.

In [1]:
# --- Environment detection + code checkout -----------------------------------
import os, subprocess, sys

def detect_environment():
    if 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ:
        return 'Colab'
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'Kaggle'
    return 'Local/Other'

ENV = detect_environment()
print(f'Current environment: {ENV}')

REPO = 'ahsank/restaurant-search'
BRANCH = 'agent/campaign'   # or a feature branch to try unmerged changes

if ENV == 'Colab':
    from google.colab import userdata
    try:
        ghtoken = userdata.get('GHTOKEN')
    except Exception:          # secret not defined / not shared with this notebook
        ghtoken = ''
    root_dir = '/content'
elif ENV == 'Kaggle':
    from kaggle_secrets import UserSecretsClient
    try:
        ghtoken = UserSecretsClient().get_secret('gh-token')
    except Exception:
        ghtoken = ''
    root_dir = '/kaggle/working'
else:
    ghtoken = ''
    root_dir = None            # running from a local checkout: research/colab/ -> repo root

if root_dir is not None and not ghtoken and os.path.isdir(f'{root_dir}/restaurant-search/research'):
    # No token but a checkout is already there (uploaded zip / colab upload): use it as is.
    REPO_DIR = f'{root_dir}/restaurant-search'
    print(f'Using existing checkout at {REPO_DIR}')
elif root_dir is not None:
    REPO_DIR = f'{root_dir}/restaurant-search'
    auth = f'ahsank:{ghtoken}@' if ghtoken else ''
    url = f'https://{auth}github.com/{REPO}.git'
    subprocess.run(['rm', '-rf', REPO_DIR], check=True)
    for branch in dict.fromkeys((BRANCH, 'main')):   # try BRANCH, then main
        r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', branch, url, REPO_DIR],
                           capture_output=True, text=True)
        if r.returncode == 0:
            print(f'Cloned {REPO}@{branch} -> {REPO_DIR}')
            break
        print(f'Clone of {branch} failed: {r.stderr.replace(ghtoken, "***") if ghtoken else r.stderr}')
    else:
        raise RuntimeError('Could not clone the repo — set the GHTOKEN secret if it is private.')
else:
    REPO_DIR = os.path.abspath('../..')

for p in (f'{REPO_DIR}/research', f'{REPO_DIR}/research/search'):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(REPO_DIR)

Current environment: Colab
Cloned ahsank/restaurant-search@agent/campaign -> /content/restaurant-search


In [2]:
# --- Install GPU dependencies (run before anything imports torch) -------------
# vLLM pins its own torch; sentence-transformers / pylate then reuse that torch.
# Colab prints dependency-conflict warnings for packages we don't use (RAPIDS, ADK) — harmless.
INSTALL_VLLM = True      # Approach 7
INSTALL_PYLATE = False  # ColBERT; conflicts with vLLM, see step 2 below

import subprocess, sys
CORE_PKGS = ['sentence-transformers']
def pip_install(*args):
    """pip install; pip's own output doesn't reach the notebook, so failures are printed here."""
    p = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], capture_output=True, text=True)
    if p.returncode != 0:
        print(f'pip install {" ".join(args)} failed (exit {p.returncode}):\n{p.stderr[-3000:]}')
    return p.returncode == 0

# 1. vLLM first, so pip installs its exact torch / torchvision / transformers. (All three in one call with
#    pylate fails: pylate pins transformers<=5.3, vLLM needs >=5.10, so pip installed nothing at all.)
pip_install(*CORE_PKGS, *(['vllm'] if INSTALL_VLLM else []))
# 2. pylate (ColBERT) only if asked: it pins transformers<=5.3 and sentence-transformers==5.3, so installing
#    it downgrades them and breaks vLLM (and pylate fails to load models with the newer versions). Use
#    INSTALL_PYLATE = True only in a session without vLLM (INSTALL_VLLM = False).
if INSTALL_PYLATE:
    if INSTALL_VLLM:
        print('Skipping pylate: it conflicts with vLLM (set INSTALL_VLLM = False for a ColBERT session).')
        INSTALL_PYLATE = False
    else:
        pip_install('pylate')
if INSTALL_VLLM:
    def ensure_torchvision():
        """Safety net if vLLM's torchvision didn't install: add the release matching vLLM's torch
        (torch 2.N.P <-> torchvision 0.(N+15).P) without touching torch."""
        if subprocess.run([sys.executable, '-c', 'import torchvision'], capture_output=True).returncode == 0:
            return
        probe = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__)'], capture_output=True, text=True)
        if probe.returncode != 0:
            print('torch not importable:', probe.stderr[-500:]); return
        major, minor, patch = (int(x) for x in probe.stdout.strip().split('+')[0].split('.')[:3])
        pip_install('--no-deps', f'torchvision==0.{minor + 15}.{patch}') or pip_install(f'torchvision==0.{minor + 15}.*')
    ensure_torchvision()
mods = ['torch'] + (['torchvision', 'vllm'] if INSTALL_VLLM else []) + (['pylate'] if INSTALL_PYLATE else [])
check = subprocess.run([sys.executable, '-c', f'import importlib; print(" | ".join(m + " " + importlib.import_module(m).__version__ for m in {mods!r}))'],
                       capture_output=True, text=True)
print(check.stdout.strip() or f'IMPORT CHECK FAILED - experiments needing these will fail:\n{check.stderr[-1500:]}')
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

torch 2.14.0+cu130 | torchvision 0.29.0+cu130 | vllm 0.33.0
name, memory.total [MiB], memory.used [MiB]
NVIDIA L4, 23034 MiB, 0 MiB


In [3]:
# --- Shared setup: data, benchmark, engine helper ---------------------------
import json, time
from pathlib import Path
import pandas as pd

import engines
from evaluate import evaluate, load_queries
from lib.data import load_data, judged_restaurant_ids

restaurants, menus = load_data(judged_restaurant_ids())
QUERIES = load_queries()
print(f'{len(restaurants)} restaurants, {sum(map(len, menus.values()))} items, {len(QUERIES)} queries')

RESULTS = {}            # engine label -> evaluate() output
SEARCH_FNS = {}         # engine label -> search_fn(query) for the playground cells

def build_engine(engine, model=None, **flags):
    # engines.build_arg_parser pre-reads sys.argv to pick engine-specific flags;
    # point it at our args instead of the kernel's.
    argv = ['--engine', engine] + (['--model', model] if model else [])
    saved, sys.argv = sys.argv, ['notebook', *argv]
    try:
        args = engines.build_arg_parser('notebook').parse_args(argv)
    finally:
        sys.argv = saved
    for k, v in flags.items():
        setattr(args, k, v)
    return engines.build_search_fn(args, restaurants, menus)

def score(label, search_fn):
    t0 = time.time()
    RESULTS[label] = evaluate(search_fn)
    SEARCH_FNS[label] = search_fn
    o = RESULTS[label]['overall']
    print(f"{label}: NDCG@5={o['ndcg@5']:.4f}  Recall@10={o['recall@10']:.4f}  MRR={o['mrr']:.4f}"
          f"  ({time.time() - t0:.0f}s)")

def summary():
    rows = {k: {m: v['overall'][m] for m in ('ndcg@5', 'recall@10', 'mrr')} for k, v in RESULTS.items()}
    return pd.DataFrame(rows).T.sort_values('ndcg@5', ascending=False).round(4)

def by(key):   # key: 'by_category' or 'by_difficulty'
    return pd.DataFrame({k: {c: m['ndcg@5'] for c, m in v[key].items()} for k, v in RESULTS.items()}).round(3)

8 restaurants, 834 items, 209 queries


## Baselines (CPU, seconds)

In [4]:
score('text', build_engine('text'))
score('enhanced', build_engine('enhanced'))

text: NDCG@5=0.4564  Recall@10=0.4804  MRR=0.4789  (0s)
enhanced: NDCG@5=0.6514  Recall@10=0.5977  MRR=0.7740  (14s)


In [5]:
score('filtered', build_engine('filtered'))

filtered: NDCG@5=0.6557  Recall@10=0.6026  MRR=0.7707  (22s)


## Approach 6 — cross-encoder (GPU)

Scores every one of the 795 items per query. ~4 s/query on CPU; a few minutes for the
whole benchmark on a GPU. Swap `CE_MODEL` for any `sentence-transformers` CrossEncoder
checkpoint (e.g. `cross-encoder/ms-marco-MiniLM-L12-v2`, `BAAI/bge-reranker-base`).

In [ ]:
# CE_MODEL = 'cross-encoder/ms-marco-MiniLM-L6-v2'
CE_MODEL = 'BAAI/bge-reranker-v2-m3'
score(f'crossencoder:{CE_MODEL.split("/")[-1]}', build_engine('crossencoder', CE_MODEL))

Optional — ColBERT / SBERT for comparison (set `INSTALL_PYLATE = True` above for ColBERT).

In [ ]:
RUN_EXTRA = True
if RUN_EXTRA:
    score('sbert:all-mpnet-base-v2', build_engine('sbert', 'all-mpnet-base-v2'))
    if INSTALL_PYLATE:
        score('colbert', build_engine('colbert'))

## Approach 7 — local LLM as reader (vLLM, GPU)

One prompt per (query, restaurant): the restaurant's whole menu as a numbered list, the query
last (so vLLM's prefix cache reuses each menu across queries). The model answers with graded
`[item_number, grade]` pairs; restaurants are merged by grade, then rank.

| GPU | Suggested `LLM_MODEL` |
|---|---|
| T4 (16 GB) | `Qwen/Qwen2.5-3B-Instruct`, `microsoft/Phi-3.5-mini-instruct` |
| L4 (24 GB) | `Qwen/Qwen2.5-7B-Instruct` (default) |
| A100 (40/80 GB) | `Qwen/Qwen2.5-14B-Instruct`, `Qwen/Qwen2.5-32B-Instruct-AWQ` |

Gated models (Llama) need an `HF_TOKEN` Colab secret. To try another model, restart the
runtime (vLLM doesn't release GPU memory cleanly) and re-run the setup cells.

In [6]:
import gc, io
import llm_search

# List of LLM Models with short names
llm_models = {
    'mistral-8b' : 'mistralai/Mistral-7B-Instruct-v0.1',
    'mistral-7b' : 'mistralai/Mistral-7B-v0.1',
    'qwen2.5-7b' : 'Qwen/Qwen2.5-7B-Instruct',
    'qwen2.5-3b' : 'Qwen/Qwen2.5-3B-Instruct',
    'qwen2.5-14b' : 'Qwen/Qwen2.5-14B-Instruct',
    'qwen2.5-32b' : 'Qwen/Qwen2.5-32B-Instruct',
    'qwen2.5-32b-awq' : 'Qwen/Qwen2.5-32B-Instruct-AWQ',
    'phi-3.5-mini' : 'microsoft/Phi-3.5-mini-instruct',
    'openai-gpt-3.5' : 'gpt-3.5-turbo',
    'openai-gpt-4' : 'gpt-4',
    'openai-oss-20b' : 'openai/gpt-oss-20b',
    'openai-oss-175b' : 'openai/gpt-oss-175b',
}
def get_model(shortname: str):
    return llm_models.get(shortname, shortname)

LLM_MODEL = get_model('openai-oss-20b')
# LLM_MODEL = 'mistralai/Ministral-8B-Instruct-2410'
# LLM_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
# LLM_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
GPU_MEM = 0.85    # vLLM's share of GPU memory (lowered automatically if less is free)

if ENV == 'Colab':
    try:
        os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))   # gated models (Llama, Gemma)
    except Exception:
        pass

llm, LLM_LOADED = None, None   # the loaded vLLM engine and its model name

def unload_llm():
    """Shut down the loaded vLLM engine and free its GPU memory. Its engine runs in a separate
    process, so `del llm` alone doesn't release the memory: the engine has to be shut down first."""
    global llm, LLM_LOADED
    if llm is None:
        return
    engine = getattr(llm, 'llm_engine', None)
    for obj in (getattr(engine, 'engine_core', None), engine):
        if hasattr(obj, 'shutdown'):
            try:
                obj.shutdown()
                break
            except Exception as e:
                print(f'vLLM shutdown warning: {e}')
    llm, LLM_LOADED = None, None
    gc.collect()
    import torch
    torch.cuda.empty_cache()

def get_llm(model_name=None):
    """The vLLM engine for model_name (default LLM_MODEL): the loaded one if it's the same model,
    otherwise the old engine is unloaded and the new model loaded. If other models (e.g. a
    cross-encoder in SEARCH_FNS) hold GPU memory, vLLM gets what's free; when that is too little,
    the cached search engines are dropped (score_category rebuilds them when needed)."""
    global llm, LLM_LOADED
    name = model_name or LLM_MODEL
    if llm is not None and LLM_LOADED == name:
        return llm
    unload_llm()
    import torch
    free, total = torch.cuda.mem_get_info()
    if free / total < 0.6 and SEARCH_FNS:
        print(f'Only {free / 2**30:.1f} of {total / 2**30:.1f} GB free: dropping cached search engines')
        SEARCH_FNS.clear(); gc.collect(); torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info()
    util = min(GPU_MEM, (free - 2**30) / total)   # vLLM refuses a share larger than the free memory
    for stream in (sys.stdout, sys.stderr):       # vLLM's engine process needs real file descriptors
        try:
            stream.fileno()
        except (io.UnsupportedOperation, AttributeError):
            stream.fileno = (lambda fd: lambda: fd)(1 if stream is sys.stdout else 2)
    print(f'Loading {name} with gpu_memory_utilization={util:.2f}')
    llm = llm_search.load_llm(name, gpu_memory_utilization=util)
    LLM_LOADED = name
    return llm

get_llm()   # load LLM_MODEL now; to switch models: get_llm('other/model') or set LLM_MODEL and rerun

Loading openai/gpt-oss-20b with gpu_memory_utilization=0.85


W1108 14:58:20.441000 3047 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1108 14:58:20.442000 3047 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1108 14:58:20.443000 3047 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=3047) INFO 11-08 14:58:32 [startup.py:96] non-default args: {'max_model_len': 12288, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'openai/gpt-oss-20b'}


config.json:   0%|          | 0.00/1.81k [00:00<?, ?B/s]

(MainProcess pid=3047) INFO 11-08 14:58:50 [model.py:744] Resolved architecture: GptOssForCausalLM


model.safetensors.index.json:   0%|          | 0.00/36.4k [00:00<?, ?B/s]

... (2 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

(MainProcess pid=3047) WARNING 11-08 14:59:00 [system_utils.py:154] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(MainProcess pid=3047) INFO 11-08 15:02:48 [hf.py:736] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


GptOssForCausalLM()(
  (model): GptOssModel()(
    (embedding): VocabParallelEmbedding(num_embeddings=201088, num_embeddings_per_partition=201088, embedding_dim=2880, org_vocab_size=201088, num_embeddings_padded=201088, tp_size=1)
    (layers): ModuleList()(
      (0-23): 24 x TransformerBlock()(
        (attn): OAIAttention()(
          (rotary_emb): YaRNScalingRotaryEmbedding(head_size=64, rotary_dim=64, max_position_embeddings=4096, base=150000, is_neox_style=True)(
            (apply_rotary_emb): ApplyRotaryEmb(is_neox_style=True, enable_fp32_compute=False)
          )
          (qkv_proj): QKVParallelLinear(in_features=2880, output_features=5120, bias=True, tp_size=1, gather_output=False)
          (o_proj): RowParallelLinear(in_features=4096, output_features=2880, bias=True, tp_size=1, reduce_results=True)
          (attn): Attention(head_size=64, num_heads=64, num_kv_heads=8, scale=0.125, backend=TritonAttentionImpl)
        )
        (mlp): MLPBlock()(
          (router): GateL

In [ ]:
# Full benchmark: 209 queries x 8 restaurants = 1,672 prompts (a few minutes on an L4).
# Rankings JSONL can be replayed anywhere: evaluate.py --engine llm --rankings FILE
name = LLM_LOADED or LLM_MODEL
tag = name.split('/')[-1].lower()
RANKINGS = Path(f'research/results/llm/{tag}.jsonl')
llm_search.run(QUERIES, restaurants, menus, name, RANKINGS, llm=get_llm(name))
score(f'llm:{tag}', llm_search.replay_search_fn(RANKINGS))

209 queries x 8 restaurants = 1672 prompts


Rendering conversations:   0%|          | 0/1672 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1672/1672 [01:52<00:00, 14.81it/s, est. speed input: 64543.18 toks/s, output: 386.48 toks/s]

Generated in 131.2s (0.63s per query)
Wrote research/results/llm/qwen2.5-7b-instruct.jsonl
llm:qwen2.5-7b-instruct: NDCG@5=0.7114  Recall@10=0.7798  MRR=0.7827  (0s)


## Compare

In [ ]:
display(summary())
display(by('by_difficulty'))
by('by_category')

## Playground — try your own queries

`SEARCH_FNS` holds every engine scored above; the LLM entry replays benchmark rankings,
so the live LLM is wired in separately.

In [7]:
def show(query, k=8):
    fns = dict(SEARCH_FNS)
    if globals().get('llm') is not None:
        fns[f'llm:{LLM_LOADED.split("/")[-1].lower()} (live)'] = lambda q: llm_search.search(q, llm, restaurants, menus)
    cols = {}
    for label, fn in fns.items():
        if label.startswith('llm:') and '(live)' not in label:
            continue
        hits = fn(query)[:k]
        cols[label] = [f"{h['name']} · {restaurants[h['restaurant_id']]['name']}"
                       + (f" [{h['grade']}]" if 'grade' in h else '') for h in hits]
        cols[label] += [''] * (k - len(cols[label]))
    return pd.DataFrame(cols, index=range(1, k + 1))



In [ ]:
show('seafood italian')

In [ ]:
show('something light and vegetarian for lunch')

## Where does the LLM miss? (per-query NDCG@5, worst first)

In [ ]:
llm_key = next((k for k in RESULTS if k.startswith('llm:')), None)
other = next((k for k in RESULTS if k.startswith('crossencoder')), None)
if llm_key:
    pq = pd.DataFrame(RESULTS[llm_key]['per_query']).set_index('query_id')
    if other:
        pq[other] = pd.DataFrame(RESULTS[other]['per_query']).set_index('query_id')['ndcg@5']
    display(pq[pq.n_relevant > 0].sort_values('ndcg@5')
              [['query_text', 'category_name', 'difficulty', 'ndcg@5'] + ([other] if other else [])].head(20))

,query_text,category_name,difficulty,ndcg@5
query_id,,,,
C01-01,Mexican food with shrimp,Cuisine type inference,Medium,0.0
C02-08,fresh cheese salad,Ingredient generalization,Medium,0.0
C02-05,smoked meat sandwich,Ingredient generalization,Hard,0.0
C02-04,stone fruit dessert,Ingredient generalization,Hard,0.0
C04-02,seared fish,Cooking method synonyms,Medium,0.0
C03-04,light fresh starter,Attribute / implied meaning,Medium,0.0
C03-02,something smoky,Attribute / implied meaning,Hard,0.0
C02-10,citrus fish,Ingredient generalization,Medium,0.0
C03-07,umami dish,Attribute / implied meaning,Hard,0.0


In [ ]:
# Inspect the raw LLM answer for one query (per restaurant)
QID = 'C03-01'
row = next(json.loads(l) for l in open(RANKINGS) if json.loads(l).get('query_id') == QID)
print(row['query_text'])
for rid, text in row['raw'].items():
    print(f'  {rid:<24} {text[:200]}')

spicy shrimp
  anthonys-homeport        [[2, 2], [39, 1]]
  bella-italia             []
  due-cucina               []
  great-china-restaurant   [[115, 1], [116, 1], [117, 1]]
  juanita-cantina          [[90, 2], [39, 1], [91, 1]]
  mendocino-farms          []
  plaza-garcia             [[87, 1], [107, 2], [118, 2]]
  staple-and-fancy         [[2, 1], [23, 1]]


## Save results

Download `research/results/` (or copy it to Drive) to keep rankings and metrics:
commit the `.jsonl` files so others can replay them with `evaluate.py --engine llm --rankings`.

In [8]:
# --- Score one category, or all of them, for one engine; terminal-style output ---
from collections import defaultdict
from evaluate import evaluate, load_queries, load_relevance, print_report

JUDGMENTS = load_relevance()   # {(restaurant_id, query_id): {item_name: relevance}}

AGENT_DIR = Path('research/agent')

def _is_experiment(name):
    return bool(name) and (AGENT_DIR / 'results' / f'{name}.json').is_file()

def find_rankings(name=None):
    """Rankings JSONL for `name`: an agent experiment name (its --rankings, e.g. 'llm-gemma-3-12b-it'),
    a file name in research/agent/runs/ or research/results/llm/ (e.g. 'gemma-3-12b-it'), or a path.
    None = this notebook's own full run (RANKINGS)."""
    if name is None:
        if 'RANKINGS' not in globals():
            raise RuntimeError('no full run in this notebook yet: pass model=<experiment or rankings file>')
        return RANKINGS
    if _is_experiment(name):
        args = json.loads((AGENT_DIR / 'results' / f'{name}.json').read_text())['eval_args']
        if '--rankings' in args:
            return Path(args[args.index('--rankings') + 1])
    stem = name[:-len('.jsonl')] if name.endswith('.jsonl') else name
    tag = stem.split('/')[-1].lower()               # a Hugging Face id -> its rankings file name
    for path in (Path(name), *(d / f'{t}.jsonl' for t in dict.fromkeys((stem, tag))
                               for d in (AGENT_DIR / 'runs', Path('research/results/llm')))):
        if path.is_file():
            return path
    return None

def experiment_fn(name):
    """search_fn for research/agent/results/<name>.json, built with the agent's exact --engine, --model,
    --rankings and flags (cached in SEARCH_FNS under the experiment's name, so two hybrids that share
    a reranker don't overwrite each other)."""
    if name not in SEARCH_FNS:
        argv = json.loads((AGENT_DIR / 'results' / f'{name}.json').read_text())['eval_args']
        saved, sys.argv = sys.argv, ['notebook', *argv]
        try:
            args, _ = engines.build_arg_parser('notebook').parse_known_args(argv)
        finally:
            sys.argv = saved
        SEARCH_FNS[name] = engines.build_search_fn(args, restaurants, menus)
    return SEARCH_FNS[name]

def _search_fn(engine, model, queries, live):
    if _is_experiment(model):                       # any agent experiment: LLM, hybrid, sbert, ...
        return experiment_fn(model)
    if engine == 'llm':
        hf_id = model is None or ('/' in model and not Path(model).exists())
        if live and hf_id:
            # Run an LLM now: model is a Hugging Face id, or None for LLM_MODEL. get_llm() unloads a
            # different loaded model first, so switching models needs no restart.
            rows = llm_search.rank_queries(get_llm(model), queries, restaurants, menus)   # one batch
            by_text = {r['query_text']: r['results'] for r in rows}
            return lambda q: by_text[q]
        path = find_rankings(model)
        if path is None:
            files = sorted(f.stem for f in (AGENT_DIR / 'runs').glob('*.jsonl'))
            raise FileNotFoundError(f'no rankings for {model!r}; experiments are in research/agent/results/, '
                                    f'files in research/agent/runs/: {", ".join(files)}')
        return llm_search.replay_search_fn(path)
    key = f'{engine}:{model}' if model else engine
    if key not in SEARCH_FNS:
        SEARCH_FNS[key] = build_engine(engine, model)
    return SEARCH_FNS[key]

def _avg(rows, key):
    rows = [r for r in rows if r['n_relevant'] > 0]
    return sum(r[key] for r in rows) / len(rows) if rows else 0.0

def _print_query_table(rows):
    w = max(len(r['query_text']) for r in rows)
    print(f"  {'query':<7} {'D':<1} {'NDCG@5':>6} {'R@10':>6} {'MRR':>6} {'#rel':>4}  text")
    print(f"  {'-'*7} {'-'} {'-'*6} {'-'*6} {'-'*6} {'-'*4}  {'-'*w}")
    for r in rows:
        print(f"  {r['query_id']:<7} {r['difficulty'][0]} {r['ndcg@5']:6.3f} {r['recall@10']:6.3f} "
              f"{r['mrr']:6.3f} {r['n_relevant']:4d}  {r['query_text']}")
    print()

def _print_details(rows, search_fn, k):
    # top-k per query: [2]/[1] = judged relevant, [ ] = not; then judged items it missed
    for r in rows:
        if r['n_relevant'] == 0:
            continue
        qid = r['query_id']
        rels = {(rid, name): rel for (rid, q), items in JUDGMENTS.items() if q == qid
                for name, rel in items.items()}
        hits = search_fn(r['query_text'])
        print(f"  {qid}  NDCG@5={r['ndcg@5']:.3f}  «{r['query_text']}»")
        for i, h in enumerate(hits[:k], 1):
            rel = rels.get((h['restaurant_id'], h['name']), 0)
            print(f"     {i}. [{rel or ' '}] {h['name']} · {restaurants[h['restaurant_id']]['name']}")
        shown = {(h['restaurant_id'], h['name']) for h in hits[:k]}
        missed = sorted(((rel, n, rid) for (rid, n), rel in rels.items() if (rid, n) not in shown),
                        reverse=True)
        for rel, n, rid in missed[:5]:
            print(f"        missed [{rel}] {n} · {restaurants[rid]['name']}")
        print()

def score_category(engine, category=None, difficulty=None, model=None,
                   live=True, details=True, k=5, report=None, return_results=False):
    """
    engine: 'text', 'enhanced', 'sbert', 'colbert', 'crossencoder', or 'llm'
    category: e.g. 'C01'; None or 'all' = every category, printed one section per category
    difficulty: 'Easy' / 'Medium' / 'Hard' (None = all)
    model: model name for sbert / colbert / crossencoder. Or an agent experiment name (a file in
           research/agent/results/, e.g. 'llm-gemma-3-12b-it' or a hybrid): rebuilt with the agent's exact
           flags, whatever `engine` says. For 'llm', also a rankings file in research/agent/runs/
           ('gemma-3-12b-it') or a path, replayed.
    live: for 'llm' when model isn't an experiment or rankings file: True = run the LLM now (model =
          a Hugging Face id, default LLM_MODEL; a different loaded model is unloaded first),
          False = replay a rankings file (no model: this notebook's full run, RANKINGS)
    details: print each query's top-k results and missed relevant items
    report: print evaluate.py's by-category / by-difficulty breakdown at the end
            (default: on for all categories, off for a single one)
    """
    if category == 'all':
        category = None
    if report is None:
        report = category is None

    queries = load_queries(category=category, difficulty=difficulty)
    if not queries:
        raise ValueError(f'no queries for category={category!r} difficulty={difficulty!r}')

    search_fn = _search_fn(engine, model, queries, live)
    res = evaluate(search_fn, category=category, difficulty=difficulty)

    o = res['overall']
    label = model if _is_experiment(model) else engine + (f':{model}' if model else '')
    print('=' * 78)
    print(f"{label}  category={category or 'all'}  difficulty={difficulty or 'all'}  "
          f"({o['n_with_judgments']}/{o['n_queries']} queries judged)")
    print(f"  NDCG@5={o['ndcg@5']:.4f}  Recall@10={o['recall@10']:.4f}  MRR={o['mrr']:.4f}")
    print('=' * 78 + '\n')

    by_cat = defaultdict(list)
    for r in res['per_query']:
        by_cat[r['category_id']].append(r)

    for cat in sorted(by_cat):
        rows = sorted(by_cat[cat], key=lambda r: r['ndcg@5'])   # worst first
        if len(by_cat) > 1:
            print('-' * 78)
            print(f"{cat}  {rows[0]['category_name']}  (n={len(rows)})  "
                  f"NDCG@5={_avg(rows, 'ndcg@5'):.3f}  Recall@10={_avg(rows, 'recall@10'):.3f}  "
                  f"MRR={_avg(rows, 'mrr'):.3f}")
            print('-' * 78)
        _print_query_table(rows)
        if details:
            _print_details(rows, search_fn, k)

    if report:
        print_report(res)
    return res if return_results else None
                      # every


## Why did the LLM grade an item this way?

`llm_explain(query, restaurant, item)` shows the item's line exactly as it appears in the LLM's prompt
(one prompt per restaurant: its whole menu, then the query), the model's answer for that restaurant
(a reasoning model's thinking and its final answer separately), the parsed grades, and the item's
benchmark label. `live=True` runs the model now (loading it if needed); `live=False` reads the answer
from a saved rankings file or experiment.

In [17]:
def _find_restaurant(name):
    key = name.lower()
    hits = [rid for rid, r in restaurants.items() if key in (rid.lower(), r.get('name', '').lower())] \
        or [rid for rid, r in restaurants.items() if key in rid.lower() or key in r.get('name', '').lower()]
    if len(hits) != 1:
        raise ValueError(f'{name!r} matches {len(hits)} restaurants: {[restaurants[r]["name"] for r in hits] or list(restaurants)}')
    return hits[0]

def llm_explain(query, restaurant, item=None, model=None, live=True, max_tokens=None,
                reasoning_effort=None, show_prompt=False, max_reasoning_chars=4000,
                system_prompt=None, show_raw=False):
    """
    query: the search text, e.g. 'cajun style shrimp' (a benchmark query or your own)
    restaurant: name or id, e.g. 'Plaza Garcia'; item: a menu item name (optional)
    model: live=True -> Hugging Face id (default: the loaded model / LLM_MODEL);
           live=False -> an experiment or rankings file, as in score_category (default: RANKINGS)
    max_tokens: default 8192 for gpt-oss (it reasons first), else 768 as in the benchmark runs
    reasoning_effort: 'low' / 'medium' / 'high' for gpt-oss (default: the model's, medium)
    show_prompt: also print the whole prompt (system prompt + menu + query)
    system_prompt: replaces llm_search.SYSTEM_PROMPT for this call (live only; the saved runs used the default)
    show_raw: also print the model's output exactly as generated, before it is split and parsed
    """
    rid = _find_restaurant(restaurant)
    entries = llm_search.menu_entries(menus[rid])
    block = llm_search.menu_block(restaurants[rid], entries)
    messages = llm_search.build_messages(query, block)
    if system_prompt is not None:
        if not live:
            raise ValueError('system_prompt needs live=True: saved answers were generated with the default prompt')
        messages[0] = {'role': 'system', 'content': system_prompt}
    numbers = {}
    if item:
        key = item.lower()
        numbers = {i: e['name'] for i, e in enumerate(entries, 1) if e['name'].lower() == key} \
            or {i: e['name'] for i, e in enumerate(entries, 1) if key in e['name'].lower()}
        if not numbers:
            raise ValueError(f'{item!r} is not on the {restaurants[rid]["name"]} menu')
        lines = block.split('\n')
        print('Item as the LLM sees it:')
        for n in numbers:
            print('  ' + next(l for l in lines if l.startswith(f'{n}. ')))
        print()
    if show_prompt:
        for m in messages:
            print(f'--- {m["role"]} ---\n{m["content"]}\n')

    if live:
        engine = get_llm(model)
        tokens = max_tokens or (8192 if 'gpt-oss' in LLM_LOADED.lower() else 768)
        t0 = time.time()
        text = llm_search._complete(engine, [messages], tokens, reasoning_effort)[0]
        source = f'{LLM_LOADED}, live, {time.time() - t0:.1f}s, max_tokens={tokens}' \
                 + (f', reasoning_effort={reasoning_effort}' if reasoning_effort else '') \
                 + (', custom system prompt' if system_prompt is not None else '')
    else:
        path = find_rankings(model)
        if path is None:
            raise FileNotFoundError(f'no rankings for {model!r}')
        row = next((json.loads(l) for l in open(path, encoding='utf-8')
                    if json.loads(l).get('query_text') == query), None)
        if row is None:
            raise ValueError(f'{query!r} is not a benchmark query in {path}; use live=True for your own queries')
        text, source = row['raw'][rid], f'saved answer in {path}'

    if show_raw:
        print(f'Raw output ({len(text)} chars):\n{text}\n')
    answer = llm_search.strip_reasoning(text)
    reasoning = text[:len(text) - len(answer)] if answer != text else ''
    for marker in llm_search._ANSWER_MARKERS:
        if reasoning.endswith(marker):
            reasoning = reasoning[:-len(marker)]
            break
    print(f'LLM output ({source}):')
    if reasoning and show_raw:
        print('  [final answer; the reasoning is in the raw output above]')
    elif reasoning:
        cut = len(reasoning) > max_reasoning_chars
        print(f'  [reasoning, {len(reasoning)} chars{", first part" if cut else ""}]')
        print('  ' + reasoning[:max_reasoning_chars].replace('\n', '\n  ') + (' ...' if cut else ''))
        print('  [final answer]')
    elif text.lstrip().startswith(llm_search._REASONING_STARTS):
        print('  [reasoning never reached a final answer: raise max_tokens or lower reasoning_effort]')
    print('  ' + (answer if answer.strip() else '(empty)').replace('\n', '\n  '))

    pairs = llm_search.parse_response(text, len(entries))
    print(f'\nParsed grades ({len(pairs)} items):')
    for n, g in pairs:
        print(f'  {"->" if n in numbers else "  "} [{g}] {n}. {entries[n - 1]["name"]}')
    if item:
        grades = dict(pairs)
        qid = next((q['query_id'] for q in QUERIES if q['query_text'] == query), None)
        label = load_relevance().get((rid, qid), {}) if qid else {}
        for n, name in numbers.items():
            judged = f'benchmark label {label.get(name, 0)} ({qid})' if qid else 'not a benchmark query'
            print(f'\n{name}: LLM grade {grades.get(n, 0)}, {judged}')

# e.g. after get_llm('openai/gpt-oss-20b'):
# llm_explain('cajun style shrimp', 'Plaza Garcia', 'Camarones a la Diabla')
# llm_explain('cajun style shrimp', 'Plaza Garcia', 'Camarones a la Diabla', model='gpt-oss-20b', live=False)   # saved run

In [24]:
# crispy fried chicken, Mendocino Farms
# cajun style shrimp, Plaza Garcia
# smoked meat sandwich
llm_explain('smoked meat sandwich', 'Mendocino Farms', system_prompt="""
Reasoning: low

You are a menu search engine.
This is a simple classification and ranking task.
Do not perform extended reasoning.
Evaluate each menu item once.
Do not reconsider previously evaluated items.
Return the result immediately after evaluating the menu.

INPUT:

One restaurant's menu containing numbered items.

A diner's natural-language search query.


OUTPUT:
Return JSON only, using this format:

[[item_number, grade], ...]

Examples:
[[12, 2], [3, 1]]
[]

Never include explanations or markdown.
""", show_prompt=True)

--- system ---

Reasoning: low

You are a menu search engine.
This is a simple classification and ranking task.
Do not perform extended reasoning.
Evaluate each menu item once.
Do not reconsider previously evaluated items.
Return the result immediately after evaluating the menu.

INPUT:

One restaurant's menu containing numbered items.

A diner's natural-language search query.


OUTPUT:
Return JSON only, using this format:

[[item_number, grade], ...]

Examples:
[[12, 2], [3, 1]]
[]

Never include explanations or markdown.


--- user ---
RESTAURANT: Mendocino Farms
Cuisine: american, sandwiches, salads, farm-to-table

MENU:
1. Heirloom BLT — $15.95 | Heirloom tomatoes, bacon, arugula, habanero honey, mayo on toasted sourdough. | Dietary: limited-time | [Sandwiches; lunch menu]
2. Not So Fried Chicken — $14.75 | Chicken, Krispies, herb aioli, mustard pickle slaw, pickled red onion, tomato on toasted ciabatta. | [Sandwiches; lunch menu]
3. The Farm Club — $14.75 | Avocado, turkey, bacon,

### Try a different system prompt

Edit `MY_PROMPT` (it starts as a copy of the benchmark's prompt, `llm_search.SYSTEM_PROMPT`) and run the
cell: `llm_try_prompt` shows the raw output and the parsed grades for your prompt, then for the default
one, so the effect of a change is easy to see. The answer must still contain `[item_number, grade]`
pairs, or nothing is parsed. To benchmark a prompt that works, change `SYSTEM_PROMPT` in
`research/search/llm_search.py` and run the full benchmark.

In [ ]:
MY_PROMPT = llm_search.SYSTEM_PROMPT    # edit, e.g. MY_PROMPT = llm_search.SYSTEM_PROMPT + "\n- A style (e.g. Cajun) ..."

def llm_try_prompt(system_prompt, query, restaurant, item=None, model=None, compare=True, **kwargs):
    """llm_explain with `system_prompt` and its raw output; with compare=True, then the same call with
    the default prompt. kwargs: max_tokens, reasoning_effort, show_prompt, max_reasoning_chars."""
    kwargs.setdefault('show_raw', True)
    runs = [('YOUR PROMPT', system_prompt)] + ([('DEFAULT PROMPT', None)] if compare else [])
    for title, prompt in runs:
        print('=' * 30, title, '=' * 30)
        llm_explain(query, restaurant, item, model=model, system_prompt=prompt, **kwargs)
        print()

print(llm_search.SYSTEM_PROMPT)
# llm_try_prompt(MY_PROMPT, 'cajun style shrimp', 'Plaza Garcia', 'Camarones a la Diabla')

In [ ]:
summary().to_csv('research/results/summary.csv')
if ENV == 'Colab':
    !cd research && zip -qr /content/results.zip results
    from google.colab import files
    files.download('/content/results.zip')

In [ ]:
def show_menu(restaurant_name, menu_name):
  # Get restaurant id from restaurant name
  id = [r['id'] for r in restaurants.values() if r['name'] == restaurant_name]
  if len(id) == 0:
    print("Restaurant not found")
    return
  restaurant_id = id[0]
  menu = [m for m in menus[restaurant_id] if m['name'] == menu_name]
  if len(menu) == 0:
    print("Menu not found")
    return
  menu = menu[0]
  print( menu)

In [ ]:
show_menu("Mendocino Farms", 'Spicy Curried Couscous')

{'name': 'Spicy Curried Couscous', 'description': "Pearl couscous, roasted cauliflower and carrots, Mendo's signature curry spice.", 'dietary': 'vegan; vegetarian', 'price': None, 'price_raw': 'N/A', 'category': 'Deli Sides & Soups', 'menu_type': 'lunch', 'restaurant_id': 'mendocino-farms'}


In [22]:

# Examples:
score_category('llm', 'all', model='gpt-oss-20b', live=False)              # replays that experiment's rankings
# score_category('llm', 'all', live=False)    # replay this notebook's full run (RANKINGS)
# score_category('llm', 'all')                # run the loaded model now

llm:gpt-oss-20b  category=all  difficulty=all  (206/209 queries judged)
  NDCG@5=0.7709  Recall@10=0.8144  MRR=0.8327

------------------------------------------------------------------------------
C01  Cuisine type inference  (n=8)  NDCG@5=0.653  Recall@10=0.693  MRR=0.875
------------------------------------------------------------------------------
  query   D NDCG@5   R@10    MRR #rel  text
  ------- - ------ ------ ------ ----  -----------------------------
  C01-05  M  0.345  0.500  0.500   14  farm to table sandwich
  C01-08  H  0.374  0.250  1.000   35  fine dining pasta
  C01-03  M  0.405  0.227  1.000   22  Latin seafood dish
  C01-04  H  0.631  1.000  0.500   10  Tex-Mex combination plate
  C01-07  H  0.720  1.000  1.000   25  rustic Italian comfort food
  C01-01  M  0.820  0.750  1.000   36  Mexican food with shrimp
  C01-02  M  0.927  0.818  1.000   15  Italian pasta with meat sauce
  C01-06  H  1.000  1.000  1.000   12  taqueria style tacos

  C01-05  NDCG@5=0.345  «farm 

In [ ]:
score_category('colbert', 'all')

In [ ]:
# --- Any agent experiment (LLM, hybrid, ...): pass its name as model ----------------------
# Names are the files in research/agent/results/ (see research/agent/LEADERBOARD.md), e.g.
#   score_category('hybrid', 'all', model='hybrid-llama4-scout-bge-reranker-v2-m3')
def use_experiment(name):
    """Returns (engine, name) for score_category(engine, ..., model=name); kept for older cells."""
    args = json.loads((AGENT_DIR / 'results' / f'{name}.json').read_text())['eval_args']
    return args[args.index('--engine') + 1], name

In [ ]:
engine, model = use_experiment('llm-gpt-oss-20b')
score_category(engine, 'all', model=model)        # or e.g. score_category(engine, 'C10', model=model)

Replaying LLM rankings from research/agent/runs/gpt-oss-20b.jsonl


Rendering conversations:   0%|          | 0/1672 [00:00<?, ?it/s]

EngineDeadError: EngineCore encountered an issue. See stack trace (above) for the root cause.